In [55]:
import warnings
import joblib
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import balanced_accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore")

### LOAD DATA

In [56]:
# Loading the data
df = pd.read_csv("cleaned_traffic_data.csv")

# Clean target labels
df["traffic_level"] = df["traffic_level"].astype(str).str.strip().str.lower()

# Map to Binary: 'low' -> 'normal' (0), 'medium'/'high' -> 'congested' (1)
target_mapping = {"low": 0, "medium": 1, "high": 1}
df["traffic_binary"] = df["traffic_level"].map(target_mapping)

# Drop any unmapped or missing target rows
df = df.dropna(subset=["traffic_binary"]).copy()
df["traffic_binary"] = df["traffic_binary"].astype(int)

# Feature extraction: Extract hour if observation_time exists
if "observation_time" in df.columns:
    df["obs_hour"] = pd.to_datetime(df["observation_time"], errors="coerce").dt.hour

### DEFINE TARGET & FEATURES

In [57]:
# Explicitly drop target columns, leaky columns, and raw text/datetime columns
drop_cols = [
    'traffic_level',
    'traffic_binary',
    'traffic_score',
    'traffic_level_encoded',
    'date',
    'observation_time',
    'time_block',
    'day_encoded',
    'traffic_pattern_encoded',
    'congestion_location_encoded'
]
X = df.drop(columns=[c for c in drop_cols if c in df.columns], errors="ignore").copy()
y = df["traffic_binary"]

numeric_features = X.select_dtypes(include=["int64", "float64", "Int64"]).columns.tolist()
categorical_features = X.select_dtypes(include=["object", "category"]).columns.tolist()

print(f"Target Distribution:\n{y.value_counts(normalize=True).round(3)}")

Target Distribution:
traffic_binary
0    0.843
1    0.157
Name: proportion, dtype: float64


In [58]:
# Identify column types dynamically after drops
numeric_features = X.select_dtypes(include=["int64", "float64", "Int64"]).columns.tolist()
categorical_features = X.select_dtypes(include=["object", "category"]).columns.tolist()

print(f"Numeric Features: {numeric_features}")
print(f"Categorical Features: {categorical_features}")

Numeric Features: ['temperature', 'rain_chance']
Categorical Features: ['road', 'fixed_corridor', 'day', 'weather', 'traffic_pattern', 'congestion_location']


### IDENTIFY COLUMN TYPE

In [59]:
numeric_features = X.select_dtypes(include=['int64', 'float64', 'Int64']).columns.tolist()
categorical_features = X.select_dtypes(include=['object', 'category']).columns.tolist()

print(f"Numeric features ({len(numeric_features)}): {numeric_features}")
print(f"Categorical features ({len(categorical_features)}): {categorical_features}")
print(f"\nTarget distribution:\n{y.value_counts()}")

Numeric features (2): ['temperature', 'rain_chance']
Categorical features (6): ['road', 'fixed_corridor', 'day', 'weather', 'traffic_pattern', 'congestion_location']

Target distribution:
traffic_binary
0    209
1     39
Name: count, dtype: int64


### PREPROCESSING PIPELINE

In [60]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            Pipeline(
                [
                    ("imputer", SimpleImputer(strategy="median")),
                    ("scaler", StandardScaler()),
                ]
            ),
            numeric_features,
        ),
        (
            "cat",
            Pipeline(
                [
                    ("imputer", SimpleImputer(strategy="most_frequent")),
                    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
                ]
            ),
            categorical_features,
        ),
    ]
)

### TRAIN / TEST SPLIT (Stratified!)

In [61]:
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

In [62]:
# With only 248 rows, a 80/20 split gives ~50 rows for testing.
# Stratify ensures low/medium/high appear in both sets.
# Encode y for compatibility across both models (especially XGBoost)
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42, stratify=y
)

# 5-fold CV is now possible with consolidated minority instances
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Compute positive class scale weight for XGBoost
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

### MODEL 1: LOGISTIC REGRESSION (Baseline)

In [63]:
lr_pipe = Pipeline(
    [
        ("prep", preprocessor),
        (
            "clf",
            LogisticRegression(
                solver="lbfgs",
                max_iter=1000,
                C=0.5,
                class_weight="balanced",
                random_state=42,
            ),
        ),
    ]
)

In [64]:
# Cross-validation (more reliable than single split on small data)
lr_cv = cross_val_score(lr_pipe, X_train, y_train, cv=cv, scoring="f1_macro")
print(f"\n--- Logistic Regression ---")
print(f"CV F1 (macro): {lr_cv.mean():.3f} (+/- {lr_cv.std():.3f})")

lr_pipe.fit(X_train, y_train)
lr_pred = lr_pipe.predict(X_test)
print(
    classification_report(
        y_test, lr_pred, target_names=["Normal (0)", "Congested (1)"], zero_division=0
    )
)
print("Balanced Accuracy:", balanced_accuracy_score(y_test, lr_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, lr_pred))


--- Logistic Regression ---
CV F1 (macro): 0.700 (+/- 0.055)
               precision    recall  f1-score   support

   Normal (0)       0.91      0.76      0.83        42
Congested (1)       0.33      0.62      0.43         8

     accuracy                           0.74        50
    macro avg       0.62      0.69      0.63        50
 weighted avg       0.82      0.74      0.77        50

Balanced Accuracy: 0.6934523809523809
Confusion Matrix:
 [[32 10]
 [ 3  5]]


### MODEL 2: XGBOOST

In [65]:
xgb_pipe = Pipeline([
    ('prep', preprocessor),
    ('clf', xgb.XGBClassifier(
        objective="binary:logistic",
        eval_metric="logloss",
        scale_pos_weight=scale_pos_weight,  # Balances minority 'congested' class
        max_depth=3,
        n_estimators=100,
        learning_rate=0.08,
        reg_lambda=2.0,
        reg_alpha=0.5,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
    ))
])

In [66]:
# Cross-validation & Evaluation
xgb_cv = cross_val_score(xgb_pipe, X_train, y_train, cv=cv, scoring="f1_macro")
print(f"\n--- XGBoost ---")
print(f"CV F1 (macro): {xgb_cv.mean():.3f} (+/- {xgb_cv.std():.3f})")

xgb_pipe.fit(X_train, y_train)
xgb_pred = xgb_pipe.predict(X_test)
print(
    classification_report(
        y_test, xgb_pred, target_names=["Normal (0)", "Congested (1)"], zero_division=0
    )
)
print("Balanced Accuracy:", balanced_accuracy_score(y_test, xgb_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, xgb_pred))


--- XGBoost ---
CV F1 (macro): 0.737 (+/- 0.032)
               precision    recall  f1-score   support

   Normal (0)       0.93      0.90      0.92        42
Congested (1)       0.56      0.62      0.59         8

     accuracy                           0.86        50
    macro avg       0.74      0.76      0.75        50
 weighted avg       0.87      0.86      0.86        50

Balanced Accuracy: 0.7648809523809523
Confusion Matrix:
 [[38  4]
 [ 3  5]]


### Save model artifacts

In [67]:
# Save the fitted end-to-end pipeline and the label encoder
joblib.dump(lr_pipe, "traffic_binary_lr_pipeline.pkl")
joblib.dump(xgb_pipe, "traffic_binary_xgb_pipeline.pkl")
print("\nBinary pipelines saved as .pkl files successfully.")


Binary pipelines saved as .pkl files successfully.
